# Modul 16: LeNet und Transfer Learning mit Keras

    **Notebooktyp:** Übungen mit ausführlichen Lösungen  
    **Vorlesungen dieses Moduls:** LeNet mit Keras, Transfer mit Keras  
    **Erwarteter Schwierigkeitsgrad:** Fortgeschrittene Keras-Anwendung für Bilddaten  
    **Orientierungszeit:** etwa 150 bis 210 Minuten

    ## Überblick

    Sie bereiten kleine Bildtensoren vor, bauen ein LeNet-ähnliches CNN und untersuchen Feature-Map-Formen, Training und Fehlerbilder. Anschließend führen Sie ein ressourcenschonendes Transfer-Learning-Experiment mit MobileNetV2 durch.

    ## Verwendete Vorlesungsnotebooks

    Die Aufgaben wurden aus dem Inhalt beider Vorlesungen dieses Moduls abgeleitet:

    - `ML Für Anfänger - Record_Module_16A_20260723.ipynb`
- `ML Für Anfänger - Record_Module_16B_20260723.ipynb`

    ## Colab-Kompatibilität

    Dieses Notebook ist für die kostenlose Version von Google Colab ausgelegt. Die Daten sind eingebaut, synthetisch erzeugt oder öffentlich verfügbar. Modelle und Trainingsbudgets sind bewusst klein gehalten. Führen Sie die Zellen in der vorgegebenen Reihenfolge aus.

## Lernziele

    Nach der Bearbeitung sollen Sie:

    - Bilddaten in passende Tensorformen bringen und Pixelwerte normalisieren.
- Ausgabeformen von Faltung, Padding, Stride und Pooling bestimmen.
- Eine LeNet-ähnliche CNN-Architektur in Keras implementieren.
- CNNs ressourcenschonend trainieren und Lernkurven sowie Fehlerbilder analysieren.
- Dropout, Datenaugmentation und Modellgröße als Regularisierungsentscheidungen beurteilen.
- MobileNetV2 laden, eine Basis einfrieren und einen neuen Klassifikationskopf trainieren.
- Transfer- und Scratch-Ansätze mit identischen Daten und nachvollziehbaren Ressourcenkennzahlen vergleichen.

    ## Bewertete Fähigkeiten

    - Bildtensoren, Normalisierung und Feature-Map-Formen
- Conv2D, MaxPooling2D, Flatten und Dense in Keras
- CNN-Training, Lernkurven, Konfusionsmatrix und Fehlerbilder
- Bildaugmentation und Dropout
- MobileNetV2, preprocess_input, Freezing und Transfer-Kopf

## Arbeitsanweisungen

Dieses Lösungsnotebook entspricht dem Übungsnotebook Aufgabe für Aufgabe. Führen Sie es von oben nach unten aus und vergleichen Sie nicht nur Endwerte, sondern auch Vorgehen, Formprüfungen, Datenaufteilung und Interpretation. Die Kommentare erklären bewusst auch typische Fehlerquellen und methodische Entscheidungen.

- Führen Sie zuerst das gemeinsame Setup aus.
- Verändern Sie vorgegebene Splits und Seeds nur, wenn eine Aufgabe dies ausdrücklich erlaubt.
- Prüfen Sie Formen, Datentypen und Wertebereiche frühzeitig.
- Begründen Sie Modell-, Metrik- und Visualisierungsentscheidungen.
- Achten Sie auf Datenleckage und eine saubere Trennung von Training, Validierung und Test.

## Gemeinsames Setup

Führen Sie diese Zelle einmal aus, bevor Sie mit Aufgabe 1 beginnen.

In [ ]:
# TensorFlow ist in Google Colab üblicherweise bereits verfügbar.
# Der Fallback installiert nur dann eine CPU-Version, wenn der Import fehlt.
import os
import sys
import subprocess
import warnings
from pathlib import Path

try:
    import tensorflow as tf
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "tensorflow-cpu"])
    import tensorflow as tf

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RANDOM_SEED = 42
FAST_MODE = os.environ.get("COURSE_FAST", "0") == "1"
OFFLINE_MODE = os.environ.get("COURSE_OFFLINE", "0") == "1"

np.random.seed(RANDOM_SEED)
tf.keras.utils.set_random_seed(RANDOM_SEED)
warnings.filterwarnings("ignore", category=FutureWarning)

import time

from sklearn.datasets import load_digits
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.model_selection import train_test_split

digits_16 = load_digits()
images_16 = digits_16.images.astype("float32")
labels_16 = digits_16.target.astype("int64")

# Die Originalpixel liegen zwischen 0 und 16. Division durch 16 bringt
# sie in den üblichen Bereich zwischen null und eins.
images_16 = images_16 / 16.0
images_16 = images_16[..., np.newaxis]

X_train_valid_16, X_test_16, y_train_valid_16, y_test_16 = train_test_split(
    images_16,
    labels_16,
    test_size=0.20,
    stratify=labels_16,
    random_state=RANDOM_SEED,
)
X_train_16, X_valid_16, y_train_16, y_valid_16 = train_test_split(
    X_train_valid_16,
    y_train_valid_16,
    test_size=0.25,
    stratify=y_train_valid_16,
    random_state=RANDOM_SEED,
)

# Für das Transfer-Experiment verwenden wir drei Klassen. Das hält den
# Datensatz klein und die Laufzeit auf Colab Free überschaubar.
transfer_mask_16 = np.isin(labels_16, [0, 1, 2])
transfer_images_16 = images_16[transfer_mask_16]
transfer_labels_16 = labels_16[transfer_mask_16]
TX_train_valid_16, TX_test_16, Ty_train_valid_16, Ty_test_16 = train_test_split(
    transfer_images_16,
    transfer_labels_16,
    test_size=0.20,
    stratify=transfer_labels_16,
    random_state=RANDOM_SEED,
)
TX_train_16, TX_valid_16, Ty_train_16, Ty_valid_16 = train_test_split(
    TX_train_valid_16,
    Ty_train_valid_16,
    test_size=0.25,
    stratify=Ty_train_valid_16,
    random_state=RANDOM_SEED,
)

print("LeNet Train/Valid/Test:", X_train_16.shape, X_valid_16.shape, X_test_16.shape)
print("Transfer Train/Valid/Test:", TX_train_16.shape, TX_valid_16.shape, TX_test_16.shape)

print("TensorFlow-Version:", tf.__version__)
print("Schneller Validierungsmodus:", FAST_MODE)


## Aufgabe 1: Bildtensoren und CNN-Ausgabeformen vorbereiten

    Untersuchen und transformieren Sie die Digits-Bilder.

1. Bestätigen Sie Batch-, Höhen-, Breiten- und Kanalachse der Trainingsdaten.
2. Prüfen Sie Wertebereich und Datentyp und visualisieren Sie je ein Beispiel der Klassen 0, 1 und 2.
3. Implementieren Sie eine Funktion für die räumliche Ausgabegröße einer Faltung.
4. Berechnen Sie die Formen nach einer `3 x 3`-Faltung mit `valid`, mit `same` und nach `2 x 2` Max-Pooling.
5. Erzeugen Sie eine einzelne, nicht trainierte `Conv2D`-Schicht mit vier Filtern, wenden Sie sie auf fünf Bilder an und visualisieren Sie die vier Feature Maps des ersten Bildes.

> **Hinweis:** Notieren Sie Formen immer im Schema `(Batch, Höhe, Breite, Kanäle)`.

In [ ]:
# ============================================================


In [ ]:
# ============================================================
# KOMMENTIERTE MUSTERLÖSUNG: Bildtensoren und CNN-Ausgabeformen vorbereiten
#
# Ziel dieser Codezelle:
# Untersuchen und transformieren Sie die Digits-Bilder. 1. Bestätigen Sie Batch-,
# Höhen-, Breiten- und Kanalachse der Trainingsdaten. 2. Prüfen Sie Wertebereich und
# Datentyp und visualisieren Sie je ein Beispiel der Kla...
#
# Die Lösung folgt bewusst einer gut prüfbaren Schrittfolge.
# Zwischenwerte und Ausgaben machen Formen, Annahmen und Ergebnisse sichtbar.
# Die fachliche Interpretation und typische Fehlerquellen stehen in der
# ausführlichen Markdown-Reflexion direkt unter dieser Codezelle.
# ============================================================

# Das Keras-Format channels-last lautet:
# (Batch, Höhe, Breite, Kanäle).
assert X_train_16.ndim == 4
batch_size, height, width, channels = X_train_16.shape
assert (height, width, channels) == (8, 8, 1)
assert X_train_16.dtype == np.float32
assert 0.0 <= float(X_train_16.min()) <= float(X_train_16.max()) <= 1.0

print("Trainingsform:", X_train_16.shape)
print("Datentyp:", X_train_16.dtype)
print("Wertebereich:", float(X_train_16.min()), "bis", float(X_train_16.max()))

# Ein festes Beispiel pro Klasse macht die Darstellung reproduzierbar.
fig, axes = plt.subplots(1, 3, figsize=(7, 2.5))
for axis, digit in zip(axes, [0, 1, 2]):
    example_index = int(np.flatnonzero(y_train_16 == digit)[0])
    axis.imshow(X_train_16[example_index, ..., 0], cmap="gray")
    axis.set_title(f"Klasse {digit}")
    axis.axis("off")
plt.tight_layout()
plt.show()

def conv_output_size(input_size, kernel_size, stride=1, padding=0):
    # Für valid padding wird die bekannte ganzzahlige Formel benutzt.
    return (input_size + 2 * padding - kernel_size) // stride + 1

valid_height = conv_output_size(8, kernel_size=3, stride=1, padding=0)
same_height = 8  # Bei stride 1 hält Keras padding="same" die Größe.
pooled_height = conv_output_size(same_height, kernel_size=2, stride=2, padding=0)
print("Nach Conv 3x3 valid:", (valid_height, valid_height))
print("Nach Conv 3x3 same:", (same_height, same_height))
print("Nach anschließendem MaxPool 2x2:", (pooled_height, pooled_height))

# Nicht trainierte Filter erzeugen bereits Feature Maps, deren Muster
# aber nur zufällig initialisiert und noch nicht auf die Aufgabe abgestimmt sind.
feature_layer_16 = tf.keras.layers.Conv2D(
    filters=4,
    kernel_size=3,
    padding="same",
    activation="relu",
)
feature_maps_16 = feature_layer_16(X_train_16[:5], training=False)
assert feature_maps_16.shape == (5, 8, 8, 4)

fig, axes = plt.subplots(1, 4, figsize=(10, 2.5))
for channel_index, axis in enumerate(axes):
    axis.imshow(feature_maps_16[0, ..., channel_index], cmap="gray")
    axis.set_title(f"Map {channel_index + 1}")
    axis.axis("off")
plt.tight_layout()
plt.show()

### Reflexion zu Aufgabe 1

Die Batchachse darf bei der Formplanung nicht mit räumlichen Achsen verwechselt werden. Faltungen ändern je nach Padding, Kernel und Stride die Höhe und Breite und erzeugen eine neue Kanalzahl entsprechend der Filterzahl. Pooling reduziert die räumliche Auflösung, lässt die Kanalzahl jedoch unverändert. Feature Maps eines untrainierten Filters haben noch keine verlässliche semantische Bedeutung.

**Kontrollfrage:** Welche Annahme, Formprüfung oder Trennungsentscheidung war für die Korrektheit dieser Lösung besonders wichtig?

## Aufgabe 2: Eine LeNet-ähnliche Architektur in Keras bauen

    Implementieren Sie ein kompaktes LeNet-ähnliches CNN für zehn Ziffernklassen.

1. Verwenden Sie eine Eingabeform `(8, 8, 1)`.
2. Bauen Sie zwei Blöcke aus `Conv2D` mit ReLU und `MaxPooling2D`.
3. Verwenden Sie anschließend `Flatten`, eine kleine Dense-Schicht und eine zehnklassige Softmax-Ausgabe.
4. Kompilieren Sie mit `sparse_categorical_crossentropy`, Adam und Accuracy.
5. Geben Sie die Modellzusammenfassung aus, zählen Sie Parameter und prüfen Sie die Ausgabeform eines Batches.

> **Hinweis:** Verfolgen Sie die Tensorform nach jeder Schicht, bevor Sie Flatten einsetzen.

In [ ]:
# Speichern Sie das Modell als lenet_16 für die nächste Aufgabe.

# ============================================================


In [ ]:
# ============================================================
# KOMMENTIERTE MUSTERLÖSUNG: Eine LeNet-ähnliche Architektur in Keras bauen
#
# Ziel dieser Codezelle:
# Implementieren Sie ein kompaktes LeNet-ähnliches CNN für zehn Ziffernklassen. 1.
# Verwenden Sie eine Eingabeform (8, 8, 1). 2. Bauen Sie zwei Blöcke aus Conv2D mit
# ReLU und MaxPooling2D. 3. Verwenden Sie anschließend F...
#
# Die Lösung folgt bewusst einer gut prüfbaren Schrittfolge.
# Zwischenwerte und Ausgaben machen Formen, Annahmen und Ergebnisse sichtbar.
# Die fachliche Interpretation und typische Fehlerquellen stehen in der
# ausführlichen Markdown-Reflexion direkt unter dieser Codezelle.
# ============================================================

tf.keras.utils.set_random_seed(RANDOM_SEED)

lenet_16 = tf.keras.Sequential(
    [
        tf.keras.layers.Input(shape=(8, 8, 1)),
        # padding="same" erhält die kleine räumliche Auflösung zunächst.
        tf.keras.layers.Conv2D(8, kernel_size=3, padding="same", activation="relu"),
        tf.keras.layers.MaxPooling2D(pool_size=2),
        tf.keras.layers.Conv2D(16, kernel_size=3, padding="same", activation="relu"),
        tf.keras.layers.MaxPooling2D(pool_size=2),
        # Flatten wandelt jede 2 x 2 x 16-Feature-Struktur in einen Vektor um.
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(32, activation="relu"),
        # Zehn Klassen benötigen zehn Wahrscheinlichkeiten, die sich zu eins summieren.
        tf.keras.layers.Dense(10, activation="softmax"),
    ],
    name="small_lenet_digits",
)

lenet_16.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=[tf.keras.metrics.SparseCategoricalAccuracy(name="accuracy")],
)

lenet_16.summary()
batch_output_16 = lenet_16(X_train_16[:7], training=False)
assert batch_output_16.shape == (7, 10)
tf.debugging.assert_near(
    tf.reduce_sum(batch_output_16, axis=1),
    tf.ones(7),
    atol=1e-5,
)

print("Trainierbare Parameter:", lenet_16.count_params())
print("Ausgabeform:", batch_output_16.shape)

### Reflexion zu Aufgabe 2

Die räumlichen Dimensionen werden schrittweise reduziert, während die Zahl der Feature Maps steigt. Dadurch kann das Netz lokale Muster zu abstrakteren Repräsentationen kombinieren. Für ganzzahlige Klassenlabels ist sparse categorical cross-entropy passend. Eine Softmax-Ausgabe stellt eine Verteilung über die zehn sich gegenseitig ausschließenden Klassen dar.

**Kontrollfrage:** Welche Annahme, Formprüfung oder Trennungsentscheidung war für die Korrektheit dieser Lösung besonders wichtig?

## Aufgabe 3: LeNet trainieren und Fehlerbilder analysieren

    Trainieren Sie `lenet_16` ressourcenschonend.

1. Verwenden Sie Early Stopping mit Wiederherstellung der besten Gewichte.
2. Trainieren Sie höchstens 30 Epochen mit einem kleinen Batch.
3. Visualisieren Sie Trainings- und Validierungsverlust sowie Accuracy.
4. Bewerten Sie das Modell auf dem Testset und erstellen Sie eine Konfusionsmatrix.
5. Finden Sie bis zu sechs falsch klassifizierte Testbilder und zeigen Sie wahres Label, Vorhersage und maximale Wahrscheinlichkeit.
6. Nennen Sie mindestens zwei plausible Ursachen für typische Verwechslungen.

> **Hinweis:** Nutzen Sie `np.flatnonzero(predictions != labels)`, um Fehlerindizes gezielt zu finden.

In [ ]:
epochs_16 = 5 if FAST_MODE else 30

# ============================================================


In [ ]:
# ============================================================
# KOMMENTIERTE MUSTERLÖSUNG: LeNet trainieren und Fehlerbilder analysieren
#
# Ziel dieser Codezelle:
# Trainieren Sie lenet16 ressourcenschonend. 1. Verwenden Sie Early Stopping mit
# Wiederherstellung der besten Gewichte. 2. Trainieren Sie höchstens 30 Epochen mit
# einem kleinen Batch. 3. Visualisieren Sie Trainings- und...
#
# Die Lösung folgt bewusst einer gut prüfbaren Schrittfolge.
# Zwischenwerte und Ausgaben machen Formen, Annahmen und Ergebnisse sichtbar.
# Die fachliche Interpretation und typische Fehlerquellen stehen in der
# ausführlichen Markdown-Reflexion direkt unter dieser Codezelle.
# ============================================================

epochs_16 = 5 if FAST_MODE else 30

history_lenet_16 = lenet_16.fit(
    X_train_16,
    y_train_16,
    validation_data=(X_valid_16, y_valid_16),
    epochs=epochs_16,
    batch_size=32,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=5,
            restore_best_weights=True,
        )
    ],
    verbose=0,
)
history_frame_16 = pd.DataFrame(history_lenet_16.history)
print("Trainierte Epochen:", len(history_frame_16))

for metric_name, label in [("loss", "Loss"), ("accuracy", "Accuracy")]:
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(history_frame_16[metric_name], label="Training")
    ax.plot(history_frame_16[f"val_{metric_name}"], label="Validierung")
    ax.set_title(f"LeNet: {label}")
    ax.set_xlabel("Epoche")
    ax.set_ylabel(label)
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

test_loss_16, test_accuracy_16 = lenet_16.evaluate(
    X_test_16,
    y_test_16,
    verbose=0,
)
test_probabilities_16 = lenet_16.predict(X_test_16, verbose=0)
test_predictions_16 = np.argmax(test_probabilities_16, axis=1)
confusion_16 = confusion_matrix(y_test_16, test_predictions_16)

print("Testverlust:", round(float(test_loss_16), 4))
print("Testgenauigkeit:", round(float(test_accuracy_16), 4))
print("Konfusionsmatrix:\n", confusion_16)

error_indices_16 = np.flatnonzero(test_predictions_16 != y_test_16)[:6]
if len(error_indices_16) == 0:
    print("Keine Fehler in der ausgewählten Testmenge gefunden.")
else:
    fig, axes = plt.subplots(1, len(error_indices_16), figsize=(2.3 * len(error_indices_16), 2.8))
    axes = np.atleast_1d(axes)
    for axis, index in zip(axes, error_indices_16):
        confidence = float(test_probabilities_16[index].max())
        axis.imshow(X_test_16[index, ..., 0], cmap="gray")
        axis.set_title(
            f"wahr {y_test_16[index]}\nvorh. {test_predictions_16[index]}\np={confidence:.2f}"
        )
        axis.axis("off")
    plt.tight_layout()
    plt.show()

### Reflexion zu Aufgabe 3

Fehler können durch ähnliche Schreibweisen, geringe Auflösung, ungewöhnliche Strichpositionen oder zu wenige ähnliche Trainingsbeispiele entstehen. Eine hohe maximale Wahrscheinlichkeit bei einem Fehler zeigt, dass Konfidenz nicht automatisch Korrektheit bedeutet. Die Konfusionsmatrix hilft, systematische Klassenpaare zu erkennen. Einzelbilder liefern anschließend konkrete Hinweise, welche visuellen Muster problematisch sind.

**Kontrollfrage:** Welche Annahme, Formprüfung oder Trennungsentscheidung war für die Korrektheit dieser Lösung besonders wichtig?

## Aufgabe 4: Regularisierung mit Augmentation und Dropout vergleichen

    Erstellen Sie eine regulierte Alternative zum ersten LeNet-Modell.

1. Verwenden Sie eine kleine Keras-Augmentationspipeline mit Rotation und Translation, die nur im Training aktiv ist.
2. Ergänzen Sie Dropout im dichten Teil des Netzes.
3. Halten Sie die übrige Architektur und die Datenpartitionen möglichst vergleichbar.
4. Trainieren Sie mit Early Stopping.
5. Vergleichen Sie beide Modelle anhand von Parameterzahl, bester Validierungsgenauigkeit und Testgenauigkeit.
6. Entscheiden Sie vorsichtig, ob die Regularisierung in diesem Lauf geholfen hat.

> **Hinweis:** Augmentation gehört in die Modell- oder Trainingspipeline, aber nicht in die Testdaten.

In [ ]:
regularized_epochs_16 = 5 if FAST_MODE else 30

# ============================================================


In [ ]:
# ============================================================
# KOMMENTIERTE MUSTERLÖSUNG: Regularisierung mit Augmentation und Dropout vergleichen
#
# Ziel dieser Codezelle:
# Erstellen Sie eine regulierte Alternative zum ersten LeNet-Modell. 1. Verwenden
# Sie eine kleine Keras-Augmentationspipeline mit Rotation und Translation, die nur
# im Training aktiv ist. 2. Ergänzen Sie Dropout im dicht...
#
# Die Lösung folgt bewusst einer gut prüfbaren Schrittfolge.
# Zwischenwerte und Ausgaben machen Formen, Annahmen und Ergebnisse sichtbar.
# Die fachliche Interpretation und typische Fehlerquellen stehen in der
# ausführlichen Markdown-Reflexion direkt unter dieser Codezelle.
# ============================================================

regularized_epochs_16 = 5 if FAST_MODE else 30
tf.keras.utils.set_random_seed(RANDOM_SEED + 1)

augmentation_16 = tf.keras.Sequential(
    [
        # Kleine Transformationen sind für Ziffern plausibel. Zu starke
        # Rotation könnte die Klassenbedeutung verändern.
        tf.keras.layers.RandomRotation(0.05, seed=RANDOM_SEED),
        tf.keras.layers.RandomTranslation(0.08, 0.08, seed=RANDOM_SEED + 1),
    ],
    name="gentle_augmentation",
)

regularized_lenet_16 = tf.keras.Sequential(
    [
        tf.keras.layers.Input(shape=(8, 8, 1)),
        augmentation_16,
        tf.keras.layers.Conv2D(8, 3, padding="same", activation="relu"),
        tf.keras.layers.MaxPooling2D(2),
        tf.keras.layers.Conv2D(16, 3, padding="same", activation="relu"),
        tf.keras.layers.MaxPooling2D(2),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(32, activation="relu"),
        tf.keras.layers.Dropout(0.25),
        tf.keras.layers.Dense(10, activation="softmax"),
    ],
    name="regularized_lenet_digits",
)
regularized_lenet_16.compile(
    optimizer=tf.keras.optimizers.Adam(0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

regularized_history_16 = regularized_lenet_16.fit(
    X_train_16,
    y_train_16,
    validation_data=(X_valid_16, y_valid_16),
    epochs=regularized_epochs_16,
    batch_size=32,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=5,
            restore_best_weights=True,
        )
    ],
    verbose=0,
)

regularized_test_loss_16, regularized_test_accuracy_16 = regularized_lenet_16.evaluate(
    X_test_16,
    y_test_16,
    verbose=0,
)
original_best_valid_16 = float(np.max(history_frame_16["val_accuracy"]))
regularized_best_valid_16 = float(np.max(regularized_history_16.history["val_accuracy"]))

regularization_comparison_16 = pd.DataFrame(
    {
        "model": ["LeNet", "LeNet mit Augmentation und Dropout"],
        "parameters": [lenet_16.count_params(), regularized_lenet_16.count_params()],
        "best_validation_accuracy": [original_best_valid_16, regularized_best_valid_16],
        "test_accuracy": [test_accuracy_16, regularized_test_accuracy_16],
    }
)
print(regularization_comparison_16.round(4).to_string(index=False))

### Reflexion zu Aufgabe 4

Augmentation ist nur sinnvoll, wenn die Transformationen die Klassenbedeutung erhalten. Dropout kann die Abhängigkeit von einzelnen Aktivierungen reduzieren. Ein einzelner Lauf beweist jedoch keine allgemeine Überlegenheit. Die Entscheidung sollte primär auf Validierungsdaten beruhen und bei wichtigen Anwendungen über mehrere reproduzierbare Läufe oder Kreuzvalidierung stabilisiert werden.

**Kontrollfrage:** Welche Annahme, Formprüfung oder Trennungsentscheidung war für die Korrektheit dieser Lösung besonders wichtig?

## Aufgabe 5: Integration: MobileNetV2 einfrieren und fair mit einem kleinen CNN vergleichen

    Führen Sie ein kleines Transfer-Learning-Experiment für die Ziffern 0, 1 und 2 durch.

1. Schreiben Sie eine Vorverarbeitung, die `8 x 8 x 1`-Bilder auf `64 x 64 x 3` bringt.
2. Erstellen Sie ein MobileNetV2-Basismodell ohne Top. Verwenden Sie ImageNet-Gewichte, falls sie verfügbar sind, und sonst einen klar dokumentierten Fallback ohne vortrainierte Gewichte.
3. Frieren Sie die Basis vollständig ein und ergänzen Sie einen Kopf für drei Klassen.
4. Erstellen Sie zusätzlich ein kleines Scratch-CNN für dieselben drei Klassen.
5. Trainieren Sie beide Modelle mit denselben Splits und derselben maximalen Epochenzahl.
6. Vergleichen Sie beste Validierungsgenauigkeit, Testgenauigkeit, trainierbare Parameter und Inferenzzeit auf demselben Teststapel.

Halten Sie die Basis eingefroren. Fine-Tuning ist noch nicht erforderlich.

> **Hinweis:** Prüfen Sie `base_model.trainable` und zählen Sie nur trainierbare Parameter.

In [ ]:
transfer_epochs_16 = 1 if FAST_MODE else 3
transfer_batch_size_16 = 32

# ============================================================


In [ ]:
# ============================================================
# KOMMENTIERTE MUSTERLÖSUNG: Integration: MobileNetV2 einfrieren und fair mit einem kleinen CNN vergleichen
#
# Ziel dieser Codezelle:
# Führen Sie ein kleines Transfer-Learning-Experiment für die Ziffern 0, 1 und 2
# durch. 1. Schreiben Sie eine Vorverarbeitung, die 8 x 8 x 1-Bilder auf 64 x 64 x 3
# bringt. 2. Erstellen Sie ein MobileNetV2-Basismodell oh...
#
# Die Lösung folgt bewusst einer gut prüfbaren Schrittfolge.
# Zwischenwerte und Ausgaben machen Formen, Annahmen und Ergebnisse sichtbar.
# Die fachliche Interpretation und typische Fehlerquellen stehen in der
# ausführlichen Markdown-Reflexion direkt unter dieser Codezelle.
# ============================================================

transfer_epochs_16 = 1 if FAST_MODE else 3
transfer_batch_size_16 = 32

def prepare_transfer_images(images):
    # MobileNetV2 erwartet drei Kanäle. Das Graustufenbild wird daher
    # nach dem Resize in identische RGB-Kanäle kopiert.
    tensor = tf.convert_to_tensor(images, dtype=tf.float32)
    tensor = tf.image.resize(tensor, (64, 64), method="bilinear")
    tensor = tf.image.grayscale_to_rgb(tensor)
    # Die Ausgangsdaten liegen in [0,1]. preprocess_input erwartet
    # Werte auf einer 0-bis-255-Skala und bildet sie dann auf [-1,1] ab.
    return tf.keras.applications.mobilenet_v2.preprocess_input(tensor * 255.0)

transfer_train_images_16 = prepare_transfer_images(TX_train_16)
transfer_valid_images_16 = prepare_transfer_images(TX_valid_16)
transfer_test_images_16 = prepare_transfer_images(TX_test_16)

# In einer Offline-Validierung wird der Download bewusst übersprungen.
# In Colab versucht das Notebook zunächst die öffentlichen ImageNet-Gewichte.
requested_weights_16 = None if OFFLINE_MODE else "imagenet"
try:
    transfer_base_16 = tf.keras.applications.MobileNetV2(
        input_shape=(64, 64, 3),
        include_top=False,
        weights=requested_weights_16,
        pooling="avg",
    )
    transfer_weight_status_16 = "ImageNet" if requested_weights_16 else "zufällige Initialisierung"
except Exception as download_error:
    print("Vortrainierte Gewichte waren nicht verfügbar. Fallback ohne Download.")
    print("Hinweis:", type(download_error).__name__)
    transfer_base_16 = tf.keras.applications.MobileNetV2(
        input_shape=(64, 64, 3),
        include_top=False,
        weights=None,
        pooling="avg",
    )
    transfer_weight_status_16 = "zufällige Initialisierung nach Fallback"

# Freezing verhindert Updates in der Basis. Nur der neue Kopf lernt.
transfer_base_16.trainable = False
transfer_inputs_16 = tf.keras.Input(shape=(64, 64, 3))
transfer_features_16 = transfer_base_16(transfer_inputs_16, training=False)
transfer_outputs_16 = tf.keras.layers.Dense(3, activation="softmax")(transfer_features_16)
transfer_model_16 = tf.keras.Model(
    transfer_inputs_16,
    transfer_outputs_16,
    name="mobilenetv2_digits_012",
)
transfer_model_16.compile(
    optimizer=tf.keras.optimizers.Adam(0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

# Ein bewusst kleines Scratch-CNN erhält dieselben vorbereiteten Bilder.
# Es startet ohne extern gelerntes Wissen.
tf.keras.utils.set_random_seed(RANDOM_SEED + 2)
scratch_model_16 = tf.keras.Sequential(
    [
        tf.keras.layers.Input(shape=(64, 64, 3)),
        tf.keras.layers.Rescaling(1.0 / 2.0, offset=0.5),
        tf.keras.layers.Conv2D(8, 5, strides=2, activation="relu"),
        tf.keras.layers.MaxPooling2D(2),
        tf.keras.layers.Conv2D(16, 3, strides=2, activation="relu"),
        tf.keras.layers.GlobalAveragePooling2D(),
        tf.keras.layers.Dense(3, activation="softmax"),
    ],
    name="scratch_digits_012",
)
scratch_model_16.compile(
    optimizer=tf.keras.optimizers.Adam(0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

transfer_history_16 = transfer_model_16.fit(
    transfer_train_images_16,
    Ty_train_16,
    validation_data=(transfer_valid_images_16, Ty_valid_16),
    epochs=transfer_epochs_16,
    batch_size=transfer_batch_size_16,
    verbose=0,
)
scratch_history_16 = scratch_model_16.fit(
    transfer_train_images_16,
    Ty_train_16,
    validation_data=(transfer_valid_images_16, Ty_valid_16),
    epochs=transfer_epochs_16,
    batch_size=transfer_batch_size_16,
    verbose=0,
)

transfer_test_accuracy_16 = transfer_model_16.evaluate(
    transfer_test_images_16,
    Ty_test_16,
    verbose=0,
)[1]
scratch_test_accuracy_16 = scratch_model_16.evaluate(
    transfer_test_images_16,
    Ty_test_16,
    verbose=0,
)[1]

def trainable_parameter_count(model):
    return int(sum(np.prod(variable.shape) for variable in model.trainable_weights))

def median_inference_seconds(model, batch, repetitions=3):
    # Ein Aufwärmlauf reduziert Initialisierungseinflüsse.
    _ = model(batch, training=False)
    durations = []
    for _ in range(repetitions):
        start_time = time.perf_counter()
        _ = model(batch, training=False)
        durations.append(time.perf_counter() - start_time)
    return float(np.median(durations))

timing_batch_16 = transfer_test_images_16[:32]
transfer_time_16 = median_inference_seconds(transfer_model_16, timing_batch_16)
scratch_time_16 = median_inference_seconds(scratch_model_16, timing_batch_16)

transfer_comparison_16 = pd.DataFrame(
    {
        "model": ["MobileNetV2-Basis", "kleines Scratch-CNN"],
        "initialization": [transfer_weight_status_16, "zufällig"],
        "best_validation_accuracy": [
            max(transfer_history_16.history["val_accuracy"]),
            max(scratch_history_16.history["val_accuracy"]),
        ],
        "test_accuracy": [transfer_test_accuracy_16, scratch_test_accuracy_16],
        "trainable_parameters": [
            trainable_parameter_count(transfer_model_16),
            trainable_parameter_count(scratch_model_16),
        ],
        "median_inference_seconds_32": [transfer_time_16, scratch_time_16],
    }
)
print(transfer_comparison_16.round(5).to_string(index=False))

### Reflexion zu Aufgabe 5

Transfer Learning nutzt eine vortrainierte Merkmalsbasis und lernt zunächst nur einen kleinen neuen Kopf. Bei stark vom Quellgebiet abweichenden Daten, hier sehr kleinen Ziffernbildern statt natürlicher Fotos, ist ein Leistungsgewinn nicht garantiert. Der Vergleich muss deshalb sowohl Daten, Trainingsbudget und Metrik als auch Parameterzahl und Inferenzzeit dokumentieren. Wird der Offline-Fallback verwendet, ist das Modell kein echtes vortrainiertes Transfermodell, auch wenn die Architektur gleich bleibt.

**Kontrollfrage:** Welche Annahme, Formprüfung oder Trennungsentscheidung war für die Korrektheit dieser Lösung besonders wichtig?

## Abschluss und Selbstkontrolle

Prüfen Sie nach dem Durcharbeiten, ob Sie jede Lösung ohne bloßes Kopieren erklären könnten. Achten Sie besonders auf die Stellen, an denen Datenleckage, unpassende Formen, falsche Metriken oder unkontrollierte Zufälligkeit zu scheinbar guten, aber methodisch falschen Ergebnissen führen könnten.

- Alle Aufgaben und Unterpunkte wurden bearbeitet.
- Verwendete Seeds und Datenpartitionen sind nachvollziehbar.
- Testdaten wurden nicht vorzeitig für Entscheidungen genutzt.
- Ergebnisse werden vorsichtig und fachlich begründet interpretiert.
- Es gibt keine hardcodierten lokalen Dateipfade oder privaten Zugangsdaten.